# Pipeline Output Analysis & Visualization

## Purpose
Post-processing / reporting stage. This notebook deliberately runs **outside the real-time pipeline** - it reads the persisted output (`classified_packets.csv`) and visualizes the distribution of MITRE ATT&CK tactics over time.

This separation mirrors a real SOC: the detection pipeline runs continuously, analysts run analytics against persisted detection output.

## What this notebook does NOT do
* No Kafka, no Jaeger, no ML training.
* Pure offline pandas + matplotlib.

## Suggested experiments
* Change the time bucket (1 min → 5 min → 1 hour).
* Group by `host` or `user` instead of `mitre_tactic` to see attacker-side patterns.
* Filter the dataframe to only failed logins to see brute-force trends.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "classified_packets.csv"
df = pd.read_csv(CSV_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["time_bucket"] = df["timestamp"].dt.floor("1min")

pivot = (
    df.groupby(["time_bucket", "mitre_tactic"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

fig, ax = plt.subplots(figsize=(18, 8))

pivot.plot(
    kind="bar",
    stacked=True,
    width=0.8,
    ax=ax,
)

ax.set_title("Cybersecurity Events Over Time (MITRE ATT&CK Tactics)")
ax.set_xlabel("Time")
ax.set_ylabel("Number of Events")

ax.legend(
    title="MITRE Tactic",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
)

plt.tight_layout()
plt.show()